## **Gdańsk vehicle feed**

In [9]:
import requests

url = "https://ckan2.multimediagdansk.pl/gpsPositions?v=2"
response = requests.get(url, timeout=20)
print(response.status_code)

200


In [10]:
data = response.json()
print(data.keys())
print(data["lastUpdate"])
print(len(data["vehicles"]))
print(data["vehicles"][0])

dict_keys(['lastUpdate', 'vehicles'])
2026-11-11T19:45:29Z
158
{'generated': '2026-11-11T19:45:27Z', 'routeShortName': '111', 'tripId': 102, 'routeId': 111, 'headsign': 'Plac Solidarności', 'vehicleCode': '2775', 'vehicleService': '111-01', 'vehicleId': 145711, 'speed': 40, 'direction': 225, 'delay': -8, 'scheduledTripStartTime': '2026-11-11T19:16:00Z', 'lat': 54.34164047241211, 'lon': 18.65998077392578, 'gpsQuality': 3}


Fetched 11 Nov 11:52 Warsaw time: 184 vehicles, JSON, no API key needed.

- `lastUpdate` - time of the whole snapshot (UTC)
- `vehicles` - list of vehicles assigned to a ZTM Gdańsk transit task in the TRISTAR system. each object contains the following fields:
    - `generated` - timestamp indicating when the vehicle's gps position was recorded (UTC)
    - `routeShortName` - public line code served by the vehicle; routeShortName value from the "Lines List" resource; string
    -`routeId` - identifier of the line to which the route variant belongs; routeId value from the "Lines List" resource; integer
    - `tripId`- public code of the variant/route along which the vehicle is traveling; tripId value from the "Routes List" resource; integer
    - `headsign` - destination/direction of the active trip; most commonly the passenger-facing name of the final stop
    - `vehicleCode` - fleet/side number of the vehicle operating the trip; string
    - `vehicleService` - The vehicle's assigned duty/shift code (matching "busServiceName" in the Timetable resource), formatted as a string "xxx-yy": xxx: Line ID ("routeId" from the Lines List resource), zero-padded to 3 digits (e.g., 008) yy:  Brigade or block number, zero-padded to 2 digits (e.g., 03)
    - `vehicleId` - internal unique identifier of public transport vehicles in the TRISTAR system; integer
    - `delay` - delay time in seconds. A negative value indicates that the vehicle is running ahead of schedule; integer, -60 means one minute earlier, +120 means 2 minutes late
    - `lat` - latitude of the vehicle's GPS position;
    - `lon` - longitude of the vehicle's GPS position;
    - `speed` - speed of the vehicle at the moment the delay was estimated; given in km/h 
    - `direction`- geographic heading/travel direction ranging from 0 to 315 in increments of 45, where 0 represents North and 315 represents Northwest 
    - `gpsQuality`- quality of the recorded GPS signal. Possible values: 0: No GPS signal, 1: Number of visible satellites insufficient for 2D mode, 2: GPS receiver operating in 2D mode, 3: GPS receiver operating in 3D mode
    - `scheduledTripStartTime` - scheduled start time of the trip, (UTC); 
   
    
    
The delay between the data in the json annd real world is 20s

In [ ]:
import time
from datetime import datetime, timezone

for i in range(3):
    data = requests.get(url, timeout=20).json()
    now = datetime.now(timezone.utc)
    ages = []

    for v in data["vehicles"]:
        # 1. Clean the string
        cleaned_string = v["generated"].replace("Z", "+00:00")
        # 2. Parse into a datetime object
        vehicle_time = datetime.fromisoformat(cleaned_string)
        # 3. Calculate difference
        time_difference = now - vehicle_time
        # 4. Convert difference to seconds (float)
        age_in_seconds = time_difference.total_seconds()
        ages.append(age_in_seconds)
    print(
        f"lastUpdate={data['lastUpdate']} vehicles={len(ages)}"
        f"newest={min(ages):.0f}s oldest={max(ages):.0f}s"
    )
    # 1. Extract the raw ISO string
    raw_timestamp = data["lastUpdate"]
    # 2. Fix 'Z' for older Python versions
    cleaned_timestamp = raw_timestamp.replace("Z", "+00:00")
    # 3. Parse into a timezone-aware datetime object
    update_time = datetime.fromisoformat(cleaned_timestamp)
    # 4. Calculate the time difference
    snapshot_age = (
        now - update_time
    )  # it calculates how many time have passed since the json was calculated and now
    print(snapshot_age.total_seconds())
    time.sleep(30)

lastUpdate=2026-11-11T20:46:18Z vehicles=144newest=7s oldest=93s
7.097668
lastUpdate=2026-11-11T20:46:39Z vehicles=144newest=16s oldest=25s
16.404211
lastUpdate=2026-11-11T20:47:19Z vehicles=142newest=6s oldest=15s
6.712761


Log refreshes every ~20–40s.  \
Lags are around ~15-25s.  \
Some vehicles stop updating (stale positions should be filtered).    \

## **Gdańsk trip updates feed**

In [ ]:
from google.transit import gtfs_realtime_pb2

tu_url = "https://ckan2.multimediagdansk.pl/gtfs-rt?feed=tripUpdates"
feed = (
    gtfs_realtime_pb2.FeedMessage()
)  # feed is a varible of class FeedMessage, generated by the protobuf compiler
feed.ParseFromString(
    requests.get(tu_url, timeout=20).content
)  # turns binary data sinto usual numbers/strings/list and attaches them to attributes

# print(feed.header)
print(feed)
# print(len(feed.entity))
# print(feed.entity[0])

header {
  gtfs_realtime_version: "2.0"
  incrementality: FULL_DATASET
  timestamp: 1794405415
}
entity {
  id: "127202611111435_202_127-08"
  trip_update {
    trip {
      trip_id: "127202611111435_202_127-08"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 15
      arrival {
        delay: 72
      }
      stop_id: "1594"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 16
      arrival {
        delay: 106
      }
      stop_id: "1634"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 17
      arrival {
        delay: 72
      }
      stop_id: "1441"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 18
      arrival {
        delay: 40
      }
      stop_id: "1754"
      schedule_relationship: SCHEDULED
    }
    stop_time_update {
      stop_sequence: 19
      arrival {
        delay: 72
      }
      stop_id: "1728"
      s

Protobuf, 259 trips at 12:27, no API key, delay per upcoming stop

- `header`:
    - `gtfs_realtime_version` - version
    - `incrementality` - "FULL_DATASET" means each fetch is a complete snapshot, not just the changes
    - `timestamp` - meaning seconds since 1970 in UTC
- list of entities:
    - `entity` - a specific trip:
        - `id` - trip_id with date, start time and route?
        - `trip_update` - a list of updates:
            - `trip` info about the trip:
                - `trip_id` - same as id of the entity
                - `schedule_relationship` - "SCHEDULED", "ADDED", "CANCELED" and so on
            - `stop_time_update` - info about each stop:
                - `stop_sequence` - position of the stop along this trip (8 = 8th stop)
                - `arrival.delay` - predicted delay in seconds
                - `stop_id`, `schedule_relationship` - self explanatory

In [7]:
v = data["vehicles"][0]
print(v["routeId"], v["tripId"], v["scheduledTripStartTime"], v["vehicleService"])

matches = [
    e.id for e in feed.entity if e.id.startswith(str(v["routeId"])) and f"_{v['tripId']}_" in e.id
]
print(matches)

131 11 2026-11-11T13:35:00Z 131-01
[]


Feeds join: trip update ID = routeId + date + local start time (HHMM) + \_tripId\_ + vehicleService. \
Checked on one vehicle; verify on more later.

## Gdynia feed - delays per stop

In [8]:
gd_url = "https://api.zdiz.gdynia.pl/pt/delays"
r = requests.get(gd_url, params={"stopId": 37030}, timeout=20)
print(r.status_code)
g = r.json()
print(g.keys())
print(g["lastUpdate"])
print(len(g["delay"]))
print(g["delay"])

200
dict_keys(['lastUpdate', 'delay'])
2026-11-11 14:57:01
4
[{'id': 'T21R10022', 'delayInSeconds': 486, 'estimatedTime': '14:59', 'headsign': 'Cisowa SKM 02', 'routeId': 10022, 'tripId': 21, 'status': 'REALTIME', 'theoreticalTime': '14:51', 'timestamp': '14:55:33', 'trip': 5901394, 'vehicleCode': 3013, 'vehicleId': 1434}, {'id': 'T41R10026', 'delayInSeconds': 21, 'estimatedTime': '15:00', 'headsign': 'Cisowa Sibeliusa 03', 'routeId': 10026, 'tripId': 41, 'status': 'REALTIME', 'theoreticalTime': '15:00', 'timestamp': '14:55:21', 'trip': 5901492, 'vehicleCode': 3004, 'vehicleId': 203696}, {'id': 'T11R10204', 'delayInSeconds': -42, 'estimatedTime': '15:00', 'headsign': 'Plac Kaszubski 03', 'routeId': 10204, 'tripId': 11, 'status': 'REALTIME', 'theoreticalTime': '15:01', 'timestamp': '14:55:33', 'trip': 5901533, 'vehicleCode': 7861, 'vehicleId': 203823}, {'id': 'T31R10141', 'delayInSeconds': 18, 'estimatedTime': '15:01', 'headsign': 'Pogórze Górne 01', 'routeId': 10141, 'tripId': 31, 'sta

Fetched 11 Nov 12:53 Warsaw time, stop 37030: 2 departures, JSON, no API key.
One request per stop (~1,100 stop poles), no citywide snapshot

- `lastUpdate` is local time with no timezone
- a list with the delays:
    - `headsign` - final destination
    - `routeId` - id of the route (can be mapped with `/pt/routes`)
    - `trip`, `tripId` - two trip IDs, meaning unclear
    - `theoreticalTime`, `estimatedTime` - are HH:MM with no date
    - `delayInSeconds` - can be negative (early)
    - `status` REALTIME = live prediction
    - `id`, `status`, `timestamp` (local time), `vehicleId`, `vehicleCode` - self-explanatory

Other endpoints (found by testing, not documented): `/pt/routes`, `/pt/trips`, `/pt/stops`, `/pt/agency`, `/pt/stop_times` (38 MB+, daily at most).

## Decision
- Live data:
    - Poller collects: Gdańsk trip updates (main source, delay per stop) + Gdańsk vehicle positions, every ~30 s
    - Gdynia (`/pt/delay`): poll a chosen list of key stops (to be picked), not all 1,100
    - Fallback: if Gdynia breaks, MVP runs on Gdańsk only
- Timetable data (collect daily):
    - `routes`, `trips` and `stops` to turn IDs into line numbers, names and locations
    - `stop_times` to compare against the schedule

- SKM: still a stretch goal, not explored